In [0]:
dbutils.widgets.text("my_sql_host","35.226.28.96")
my_sql_host=dbutils.widgets.get("my_sql_host")

dbutils.widgets.text("my_sql_port","3306")
my_sql_port=dbutils.widgets.get("my_sql_port")

dbutils.widgets.text("my_sql_user","root")
my_sql_user=dbutils.widgets.get("my_sql_user")

dbutils.widgets.text("my_sql_pwd","Admin@1234")
my_sql_pwd=dbutils.widgets.get("my_sql_pwd")

dbutils.widgets.text("my_sql_db","GCPMigrationmMeta")
my_sql_db=dbutils.widgets.get("my_sql_db")

dbutils.widgets.text("sak","/Volumes/workspace/default/gcptodbmigration/sk/datamigrationproject-494118-b1bc74441af6.json")
sak=dbutils.widgets.get("sak")

dbutils.widgets.text("hive_db","bronze")
hive_db=dbutils.widgets.get("hive_db")

In [0]:
%pip install -q mysql-connector-python google-cloud-bigquery google-cloud-storage

In [0]:
import os,re,shutil,json
import mysql.connector as sql
from contextlib import contextmanager
from urllib.parse import urlparse
from google.cloud import storage
from pyspark.sql.functions import *
from pyspark.sql.types import *

MySql={
"host":my_sql_host,
"port":my_sql_port,
"user":my_sql_user,
"pwd":my_sql_pwd,
"db":my_sql_db
}

assert os.path.exists(sak), f"GCP key not found at {sak}"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"]=sak

In [0]:
@contextmanager
def my_sql_conn():
    conn=sql.connect(
        host=MySql["host"],
        port=int(MySql["port"]),
        user=MySql["user"],
        password=MySql["pwd"],
        database=MySql["db"]
    )
    try:
        yield conn
    finally:
        conn.close()

In [0]:
def fetch_eligible_rows():
    with my_sql_conn() as conn:
        cur=conn.cursor(dictionary=True)
        cur.execute("""select table_name, gcs_path, target_path
                    from config_table
                    where active_flag=1
                    and load_flag=1
                    and bq_to_gcs_status="COMPLETED"
                    and gcs_to_bronze_status in ("NOT_STARTED","FAILED")
                    order by table_name""")
        rows=cur.fetchall()
        cur.close()
    return rows

In [0]:
def set_bronze_status(table_name,status,error=None):
    with my_sql_conn() as conn:
        cur=conn.cursor()
        if status=="IN_PROGRESS":
            cur.execute("""update config_table
                        set gcs_to_bronze_status="IN_PROGRESS",
                        last_run_ts=NOW(), error_message=NULL
                        where table_name=%s""",(table_name,))
        elif status=="COMPLETED":
            cur.execute("""update config_table
                        set gcs_to_bronze_status="COMPLETED",
                        last_run_ts=NOW(), last_success_ts=NOW(),error_message=NULL
                        where table_name=%s""",(table_name,))
        else:
            cur.execute("""update config_table
                        set gcs_to_bronze_status="FAILED",
                        last_run_ts=NOW(), error_message=%s
                        where table_name=%s""",(str(error)[:2000] if error else "FAILED",table_name))
        conn.commit()
        cur.close()

In [0]:
def reset_load_flag(table_name):
    with my_sql_conn() as conn:
        cur=conn.cursor()
        cur.execute("""update config_table
                    set load_flag=0
                    where table_name=%s""",(table_name,))
        conn.commit()
        cur.close()

In [0]:
def _gcs_client():
    return storage.Client()

In [0]:
"""print(_gcs_client())"""

In [0]:
def _split_gs(gcs_uri: str):
    """split a GCS URI (gs://bucket/path) into (bucket, path) coming from config_table"""
    assert gcs_uri.startswith("gs://"), f"Invalid GCS URI: {gcs_uri}"
    p=urlparse(gcs_uri)
    return p.netloc, p.path.lstrip("/")    

In [0]:
"""print(_split_gs("gs://bigquerytogcsmigration/exports/ods/ods_order_items"))"""

In [0]:
def latest_date_uri(gcs_base : str):
    """
    Find the latest dt=YYYYMMDDTHHMMSSZ folder under gcs_base.
    Return a wildcard URI for all Parquet files in that folder."""

    bucket_name,base_prefix=_split_gs(gcs_base.rstrip("/"))
    client=_gcs_client()
    #list all objects under base_prefix/dt=
    search_prefix=f"{base_prefix}/dt="
    dts=set()
    for blob in client.list_blobs(bucket_name,prefix=search_prefix):
        # Look for '.../dt=xxxxx/' in blob.name or files under it
        m=re.search(r"dt=(\d+)/", blob.name)
        if m:
            dts.add(m.group(1))
    if not dts:
        # fallback: allow wildcard if no dt folders found
        return f"gs://{bucket_name}/{base_prefix}/dt=*/*.parquet"
    latest=sorted(dts)[-1]
    return f"gs://{bucket_name}/{base_prefix}/dt={latest}/*.parquet"

In [0]:
def _ensure_dir(local_path: str):
    """Create a local directory if it does not exist."""
    os.makedirs(local_path, exist_ok=True)

In [0]:
def _prefix_from_wildcard(gcs_uri: str) -> tuple[str, str]:
    """
    Given a GCS URI with wildcard, return (bucket, prefix) for listing.
    Example: gs://bucket/foo/bar/dt=.../*.parquet -> (bucket, 'foo/bar/dt=.../')
    """
    bucket, path=_split_gs(gcs_uri)
    if "*" in path:
        prefix=path[:path.rfind("/") + 1]
    else:
        prefix=path if path.endswith("/") else path + "/"
    return bucket, prefix


In [0]:
def copy_gcs_prefix_to_dbfs(gcs_uri_wildcard: str, dbfs_dir: str) -> str:
    """
    Copy all objects under the wildcard's parent prefix from GCS to DBFS directory.
    Returns the DBFS directory containing the downloaded Parquet files.
    """
    bucket, prefix=_prefix_from_wildcard(gcs_uri_wildcard)
    cli=_gcs_client()
    bkt=cli.bucket(bucket)

    if os.path.exists(dbfs_dir):
        shutil.rmtree(dbfs_dir)
    _ensure_dir(dbfs_dir)

    n=0
    for blob in cli.list_blobs(bucket, prefix=prefix):
        if blob.name.endswith("/"):
            continue
        local_file=os.path.join(dbfs_dir, os.path.basename(blob.name))
        print(local_file)
        blob.download_to_filename(local_file)
        n+=1
    if n==0:
        raise FileNotFoundError(f"No objects found under gs://{bucket}/{prefix}")
    print(f"↳ Copied {n} file(s) from gs://{bucket}/{prefix} → {dbfs_dir}")
    return local_file

In [0]:
rows = fetch_eligible_rows()
if not rows:
    print("Nothing to process (eligible rows not found).")
else:
    # Ensure Hive database exists for Delta tables
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {hive_db}")
    print(f"Processing {len(rows)} table(s): {[r['table_name'] for r in rows]}")
    for r in rows:
        t=r["table_name"]
        gcs_base=r["gcs_path"]
        dst_path= r["target_path"]
        try:
            set_bronze_status(t, "IN_PROGRESS")

            # 1) Pick latest dt folder in GCS for this table
            src_uri=latest_date_uri(gcs_base)
            print(f"\n{t}: staging {src_uri}")

            # 2) Copy Parquet files from GCS to DBFS staging directory
            stage_dir=f"/Volumes/workspace/default/gcptodbmigration/gcs_stage/{t}"
            stage_dir=copy_gcs_prefix_to_dbfs(src_uri, stage_dir)
            print(stage_dir)
            print("spark read start")
            # 3) Read Parquet files from DBFS and write to Delta table (overwrite mode)
            df=spark.read.parquet(stage_dir)
            print("spark read end")
            # --- Normalize time columns (minimal) ---
            # Force 'order_ts' to proper TIMESTAMP regardless of source variation
            print("transform start")
            if 'order_ts' in df.columns:
                # This safely handles string, date, long (epoch sec/ms if you already pre-converted),
                # and passes through if it's already timestamp.
                df=df.withColumn('order_ts', to_timestamp(col('order_ts')))
            print("transform end")
            print("spark overwrite start")
            df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").save(dst_path)
            df.write.mode("overwrite")\
                    .option("overwriteSchema", "true")\
                    .format("delta")\
                    .saveAsTable(f"{hive_db}.{t}")
            print("spark overwrite end")
            # 4) Register Delta table in Hive (bronze DB)
            '''print("delta table start")
            sql_query = f"""CREATE TABLE IF NOT EXISTS workspace.{hive_db}.{t}
                         USING DELTA LOCATION '{dst_path}'"""
            print(sql_query)
            spark.sql(sql_query)
            print("delta table end")'''
            set_bronze_status(t, "COMPLETED")
            reset_load_flag(t)
            print(f"✅ {t}: Bronze written at {dst_path}")
        except Exception as e:
            set_bronze_status(t, "FAILED", error=e)
            print(f"❌ {t}: {e}")

# (Optional) Clean up DBFS staging directory after run
dbutils.fs.rm("/Volumes/workspace/default/gcptodbmigration/gcs_stage", recurse=True)